# 00 · PyTorch Fundamentals: Refresher Notebook

*Part of my **PyTorch Marathon**. These are my own notes, written for future me coming back for a quick refresh.*

**How to use this notebook**
- Short explanations sit above each code cell. Sections marked 🔑 go deeper because they matter most later.
- Each section ends with a **TL;DR** for skimming.
- The **Cheat Sheet** and **Common Errors** tables at the end are the fastest way to refresh.

### Contents
1. What is PyTorch?
2. Setup
3. Tensors: scalar → vector → matrix → tensor 🔑
4. Creating tensors
5. Datatypes (dtypes) 🔑
6. Tensor attributes: shape, dtype, device
7. Basic operations and broadcasting 🔑
8. Matrix multiplication 🔑
9. `nn.Linear`: matmul inside a layer
10. Aggregation and the `dim` argument 🔑
11. Reshaping, stacking, squeezing, permuting 🔑
12. Indexing and boolean masks
13. PyTorch ↔ NumPy (and memory sharing)
14. Reproducibility (random seeds)
15. Devices: CPU, CUDA, MPS 🔑
16. Bonus: a first look at autograd
17. Cheat sheet
18. Common errors and fixes
19. Exercises

## 1. What is PyTorch?

PyTorch is an open-source deep learning framework. In practice it gives you three things:

1. **Tensors**: n-dimensional arrays (like NumPy arrays) that can live on a **GPU**.
2. **Autograd**: automatic computation of gradients, which is what makes training possible.
3. **Building blocks** (`torch.nn`, `torch.optim`, `torch.utils.data`) for models, optimisers and data pipelines.

**Why PyTorch?** It's Pythonic, *eager*: code runs line by line, so you can debug with plain `print`. It dominates research and is widely used in production, and the ecosystem (torchvision, Hugging Face, Lightning) is built around it.

> **Mental model:** every deep learning workflow is *data → tensors → operations on tensors → better tensors*.

## 2. Setup

In [ ]:
import torch
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version:   {np.__version__}")

## 3. Tensors 🔑

A **tensor** is the data structure for *everything* in PyTorch: inputs, outputs, model weights and gradients.

Its two key properties:
- **`ndim`**: the number of dimensions, or *axes*. Quick trick: count the opening square brackets `[` on one side.
- **`shape`**: the size of each dimension, read **outer → inner**.

### Real-world shapes you'll see constantly

| Data | Typical shape | Meaning |
|---|---|---|
| Single image | `[3, 224, 224]` | `[colour_channels, height, width]` (**CHW**, PyTorch's default) |
| Batch of images | `[32, 3, 224, 224]` | `[batch, C, H, W]` (**NCHW**) |
| Tabular batch | `[64, 10]` | `[batch, features]` |
| Text tokens | `[16, 128]` | `[batch, sequence_length]` |
| Text embeddings | `[16, 128, 768]` | `[batch, seq_len, embedding_dim]` |

> ⚠️ Libraries like PIL, OpenCV and Matplotlib use **HWC** (`[224, 224, 3]`). You often have to convert between HWC and CHW; see `permute` in Section 11.

### 3.1 Scalar (0-D)
A single number. `.item()` pulls it out as a plain Python number. It only works for one-element tensors, and it's handy for logging loss values.

In [ ]:
scalar = torch.tensor(7)
print(scalar, "| ndim:", scalar.ndim, "| shape:", scalar.shape)
print("As Python int:", scalar.item(), type(scalar.item()))

### 3.2 Vector (1-D)
A list of numbers, e.g. `[bedrooms, bathrooms, parking]` describing a house.

In [ ]:
vector = torch.tensor([3, 2, 1])
print(vector, "| ndim:", vector.ndim, "| shape:", vector.shape)

### 3.3 Matrix (2-D)
Rows × columns, like a spreadsheet or a weight matrix.

In [ ]:
MATRIX = torch.tensor([[7, 8],
                       [9, 10]])
print(MATRIX, "\nndim:", MATRIX.ndim, "| shape:", MATRIX.shape)

### 3.4 Tensor (n-D)
Any number of dimensions. The shape below is `[1, 3, 3]`: **one** block of a **3×3** grid.

In [ ]:
TENSOR = torch.tensor([[[1, 2, 3],
                        [3, 6, 9],
                        [2, 4, 5]]])
print(TENSOR, "\nndim:", TENSOR.ndim, "| shape:", TENSOR.shape)

### Summary table

| Name | What it is | `ndim` | Example shape | Naming convention |
|---|---|---|---|---|
| **Scalar** | A single number | 0 | `torch.Size([])` | lowercase (`a`) |
| **Vector** | 1-D list of numbers | 1 | `[3]` | lowercase (`y`) |
| **Matrix** | 2-D grid of numbers | 2 | `[2, 2]` | UPPERCASE (`W`) |
| **Tensor** | n-D array (covers all of the above) | n | `[1, 3, 3]` | UPPERCASE (`X`) |

**TL;DR:** everything is a `torch.Tensor`. What you *call* it depends on `ndim`. Always read shapes outer → inner.

## 4. Creating tensors

You rarely type tensors out by hand. Models **start with random numbers** and adjust them during training:

`random init → look at data → update numbers → look at data → update numbers → …`

| Function | What it gives you | Typical use |
|---|---|---|
| `torch.tensor(data)` | Tensor from a Python list/array (copies data) | Small, hand-made examples |
| `torch.rand(size)` | Uniform random values in `[0, 1)` | Quick test data |
| `torch.randn(size)` | Normal distribution (mean 0, std 1) | Weight-init-like values |
| `torch.randint(low, high, size)` | Random integers | Fake labels or indices |
| `torch.zeros(size)` / `torch.ones(size)` | All 0s / all 1s | Masks, initial biases |
| `torch.full(size, value)` | Filled with `value` | Constants |
| `torch.arange(start, end, step)` | Range of values (end excluded) | Indices, sequences |
| `torch.linspace(start, end, steps)` | `steps` evenly spaced values (end **included**) | Plotting, toy data |
| `torch.eye(n)` | Identity matrix | Linear algebra |
| `torch.zeros_like(t)` / `ones_like` / `rand_like` | Same shape (and dtype/device) as `t` | Matching an existing tensor |

> Use `torch.arange`, not `torch.range`. The latter is deprecated and includes the end value, which is confusing.

In [ ]:
random_tensor = torch.rand(size=(3, 4))
normal_tensor = torch.randn(3, 4)              # size can be passed positionally too
random_image  = torch.rand(size=(3, 224, 224))  # CHW image-shaped tensor

print("rand:\n", random_tensor)
print("randn:\n", normal_tensor)
print("Image-like tensor shape:", random_image.shape, "| ndim:", random_image.ndim)

In [ ]:
zeros = torch.zeros(size=(2, 3))
ones  = torch.ones(size=(2, 3))
sevens = torch.full((2, 3), 7.0)
identity = torch.eye(3)

print(zeros, ones, sevens, identity, sep="\n\n")

In [ ]:
zero_to_nine = torch.arange(0, 10)          # end is EXCLUDED -> 0..9
evens        = torch.arange(0, 10, 2)
five_points  = torch.linspace(0, 1, steps=5)  # end is INCLUDED

print(zero_to_nine)
print(evens)
print(five_points)

# *_like copies the shape (and dtype/device) of another tensor
print(torch.zeros_like(zero_to_nine))

**TL;DR:** `rand`/`randn` for random values, `zeros`/`ones`/`full` for constants, `arange`/`linspace` for ranges, `*_like` to match an existing tensor.

## 5. Datatypes (dtypes) 🔑

Every tensor has a single **dtype**. It controls **precision** (how much detail a number holds), **memory**, and **speed**.

- More bits → more precise, but more memory and slower.
- Fewer bits → faster and smaller, at a slight cost in accuracy. This is the idea behind **mixed precision** training and **quantisation** for deployment.

| dtype | Alias | Bits | Where you'll see it |
|---|---|---|---|
| `torch.float32` | `torch.float` | 32 | **Default** for floats; model weights and most training |
| `torch.float64` | `torch.double` | 64 | NumPy's default; rarely needed in DL |
| `torch.float16` | `torch.half` | 16 | Mixed-precision training on GPUs |
| `torch.bfloat16` | – | 16 | Mixed precision on modern GPUs/TPUs (same range as fp32, less precision) |
| `torch.int64` | `torch.long` | 64 | **Default** for integers; class labels, indices |
| `torch.int32` | `torch.int` | 32 | General integers |
| `torch.int8` / `torch.uint8` | – | 8 | Quantised models; raw image pixels (0–255) |
| `torch.bool` | – | – | Masks and comparison results |

### Why it matters
The three most common PyTorch bugs are **shape**, **dtype** and **device** mismatches. Typical dtype examples:
- `nn.CrossEntropyLoss` expects labels as `torch.long`.
- Feeding `float64` data (straight from NumPy) into a `float32` model raises an error.
- `.mean()` doesn't work on integer tensors.

In [ ]:
float_32_tensor = torch.tensor([3.0, 6.0, 9.0],
                               dtype=None,          # None -> inferred (float32 here)
                               device=None,         # None -> CPU by default
                               requires_grad=False) # True -> autograd tracks operations (Section 16)

int_tensor = torch.tensor([1, 2, 3])

print(float_32_tensor.dtype, "|", int_tensor.dtype)

In [ ]:
float_16_tensor = torch.tensor([3.0, 6.0, 9.0], dtype=torch.float16)

# Mixed dtypes: PyTorch "promotes" to the more precise type for simple ops
result = float_16_tensor * float_32_tensor
print(result, result.dtype)

> **Type promotion:** element-wise ops between different float dtypes usually promote to the higher precision, as above. Many layers and losses are stricter, though, so it's best to **keep dtypes consistent yourself**.

**TL;DR:** floats default to `float32`, ints to `int64`. Lower precision is faster but less precise. Mismatched dtypes are a top-3 error source.

## 6. Tensor attributes: shape, dtype, device

When something breaks, check these three first. A handy question to ask: **"What shape? What dtype? Where is it?"**

| Attribute | Answers | Example value |
|---|---|---|
| `t.shape` / `t.size()` | What shape is it? | `torch.Size([3, 4])` |
| `t.dtype` | What type are the elements? | `torch.float32` |
| `t.device` | Where is it stored? | `cpu`, `cuda:0`, `mps:0` |
| `t.ndim` | How many dimensions? | `2` |
| `t.numel()` | How many elements in total? | `12` |

In [ ]:
some_tensor = torch.rand(3, 4)

print(some_tensor)
print(f"Shape:    {some_tensor.shape}")
print(f"Dtype:    {some_tensor.dtype}")
print(f"Device:   {some_tensor.device}")
print(f"ndim:     {some_tensor.ndim}")
print(f"Elements: {some_tensor.numel()}")

## 7. Basic operations and broadcasting 🔑

Neural networks are mostly built from a handful of operations: **addition, subtraction, element-wise multiplication, division and matrix multiplication**, plus non-linear functions.

### 7.1 Element-wise arithmetic
Operators apply to **each element**. They return a **new** tensor and the original is unchanged unless you reassign it.

In [ ]:
tensor = torch.tensor([1, 2, 3])

print("+ 10:", tensor + 10)
print("* 10:", tensor * 10)
print("- 10:", tensor - 10)
print("/ 10:", tensor / 10)          # true division -> float result
print("Original unchanged:", tensor)

# Function equivalents (operators are more common)
print(torch.add(tensor, 10), torch.mul(tensor, 10))

# Element-wise multiplication of two tensors
print("tensor * tensor:", tensor * tensor)

### 7.2 In-place operations
Methods ending in an underscore (`add_`, `mul_`, `zero_`) **modify the tensor in place**. They save memory, but use them with care: they can break autograd and change tensors that share memory with others (see `view` and NumPy below).

In [ ]:
t = torch.tensor([1., 2., 3.])
t.add_(10)          # same as t += 10
print(t)

### 7.3 Broadcasting (added: important!)
Broadcasting lets PyTorch combine tensors of **different shapes** without copying data. That's how `tensor + 10` works: the scalar is "stretched" across the tensor.

**Rule:** compare the shapes **from the right**. Two dimensions are compatible if they are **equal**, or **one of them is 1** (or missing).

| Shape A | Shape B | Result | Why |
|---|---|---|---|
| `[3, 4]` | `[]` (scalar) | `[3, 4]` | The scalar broadcasts everywhere |
| `[3, 4]` | `[4]` | `[3, 4]` | `[4]` is treated as `[1, 4]`, repeated over 3 rows |
| `[3, 4]` | `[3, 1]` | `[3, 4]` | The size-1 column is repeated over 4 columns |
| `[3, 1]` | `[1, 4]` | `[3, 4]` | Both stretch, giving an "outer" combination |
| `[3, 4]` | `[3]` | ❌ Error | Rightmost dims are 4 vs 3: not equal and neither is 1 |

A classic use is normalising features: `(X - mean) / std`, where `X` is `[batch, features]` and `mean` is `[features]`.

In [ ]:
X = torch.arange(12, dtype=torch.float32).reshape(3, 4)   # [3, 4]
row = torch.tensor([10., 20., 30., 40.])                   # [4]
col = torch.tensor([[100.], [200.], [300.]])               # [3, 1]

print("X + row:\n", X + row)
print("X + col:\n", X + col)

# Feature-wise standardisation using broadcasting
X_norm = (X - X.mean(dim=0)) / X.std(dim=0)
print("Standardised X:\n", X_norm)

**TL;DR:** operators are element-wise and return new tensors. `_` methods change the tensor in place. Broadcasting matches shapes from the right, where a dimension must be equal or 1.

## 8. Matrix multiplication 🔑

Matrix multiplication (**matmul**) is *the* core operation of deep learning: linear layers, attention and convolutions (under the hood) all rely on it.

### Element-wise vs matrix multiplication
For `tensor = [1, 2, 3]`:

| Operation | Calculation | Result | Code |
|---|---|---|---|
| Element-wise | `[1*1, 2*2, 3*3]` | `[1, 4, 9]` | `tensor * tensor` |
| Matrix multiplication (dot product for 1-D) | `1*1 + 2*2 + 3*3` | `14` | `tensor @ tensor` or `torch.matmul(tensor, tensor)` |

The difference: matmul **multiplies and then sums**.

### The two rules
1. **Inner dimensions must match:** `(a, b) @ (b, c)` ✅, but `(a, b) @ (c, d)` ❌ when `b ≠ c`.
2. **The result takes the outer dimensions:** `(a, b) @ (b, c) → (a, c)`.

| Left | Right | Works? | Output shape |
|---|---|---|---|
| `(2, 3)` | `(3, 2)` | ✅ | `(2, 2)` |
| `(3, 2)` | `(2, 3)` | ✅ | `(3, 3)` |
| `(3, 2)` | `(3, 2)` | ❌ | inner dims 2 ≠ 3 |
| `(3, 2)` | `(3, 2).T` = `(2, 3)` | ✅ | `(3, 3)` |
| `(B, n, m)` | `(B, m, p)` | ✅ (batched) | `(B, n, p)` |

### Ways to call it

| Code | Notes |
|---|---|
| `A @ B` | Python operator; identical to `torch.matmul` and the most readable |
| `torch.matmul(A, B)` | General: handles 1-D, 2-D, batched, and broadcasts batch dims |
| `torch.mm(A, B)` | **2-D only**, no broadcasting |
| `torch.bmm(A, B)` | **3-D batched only**: `(B, n, m) @ (B, m, p)` |

In [ ]:
tensor = torch.tensor([1, 2, 3])

print("Element-wise:", tensor * tensor)
print("matmul:      ", torch.matmul(tensor, tensor))
print("@ operator:  ", tensor @ tensor)

### Why not write it with loops?
PyTorch's matmul calls highly optimised C++/CUDA kernels, and **vectorised operations beat Python loops** by orders of magnitude. The comparison below uses a bigger tensor so the gap is visible.

In [ ]:
import time

big = torch.rand(100_000)

start = time.perf_counter()
value = 0
for i in range(len(big)):
    value += big[i] * big[i]
loop_time = time.perf_counter() - start

start = time.perf_counter()
value_vec = big @ big
vec_time = time.perf_counter() - start

print(f"Python loop: {loop_time*1000:.2f} ms")
print(f"Vectorised:  {vec_time*1000:.4f} ms")
print(f"Same answer? {torch.isclose(value, value_vec)}")

### Shape errors: the most common error in deep learning
Two `(3, 2)` matrices can't be multiplied because the inner dims are `2` vs `3`. The fix is often a **transpose** (`.T`, or `torch.transpose(t, dim0, dim1)`), which swaps the dimensions.

In [ ]:
tensor_A = torch.tensor([[1, 2],
                         [3, 4],
                         [5, 6]], dtype=torch.float32)   # (3, 2)

tensor_B = torch.tensor([[7, 10],
                         [8, 11],
                         [9, 12]], dtype=torch.float32)  # (3, 2)

try:
    torch.matmul(tensor_A, tensor_B)
except RuntimeError as e:
    print("❌ RuntimeError:", e)

In [ ]:
print(f"tensor_A: {tensor_A.shape} | tensor_B.T: {tensor_B.T.shape}  <- inner dims now match (2 == 2)")

output = tensor_A @ tensor_B.T
print(output)
print("Output shape:", output.shape, " <- outer dims (3, 3)")

> **Debugging habit:** when you hit `mat1 and mat2 shapes cannot be multiplied`, **print the shapes** of both operands, write them side by side, and check the inner dimensions.

**TL;DR:** matmul multiplies and sums. Inner dims must match, and the output takes the outer dims. Use `@`. Fix mismatches with `.T`, `reshape` or `permute`.

## 9. `nn.Linear`: matrix multiplication inside a layer

A linear (also called *fully connected* or *dense*) layer computes:

$$ y = xW^T + b $$

| Symbol | Meaning | Shape (for `nn.Linear(in_features, out_features)`) |
|---|---|---|
| `x` | Input | `[batch, in_features]` |
| `W` | Weight matrix (starts random, gets learned) | `[out_features, in_features]` |
| `b` | Bias (learned offset) | `[out_features]` |
| `y` | Output | `[batch, out_features]` |

It's the "straight line" `y = mx + b`, generalised to many inputs and outputs.

**Shape rule:** `in_features` must equal the **last dimension** of the input.

In [ ]:
torch.manual_seed(42)  # weights are random; seed for repeatable output (Section 14)

linear = torch.nn.Linear(in_features=2, out_features=6)
x = tensor_A                                  # (3, 2): 3 samples, 2 features each

output = linear(x)
print(f"Input shape:  {x.shape}")
print(f"Weight shape: {linear.weight.shape}  (out_features, in_features)")
print(f"Bias shape:   {linear.bias.shape}")
print(f"Output shape: {output.shape}")

# Verify it's really just matmul + bias
manual = x @ linear.weight.T + linear.bias
print("Matches manual xW^T + b?", torch.allclose(output, manual))

> **Try it:** change `in_features` to `3`. You'll get a shape error, because the input's last dim is 2. How would you fix it?

## 10. Aggregation and the `dim` argument 🔑

Aggregation goes from **many values to fewer values**: min, max, mean, sum.

> ⚠️ `mean()` (and `std()`) need a **floating-point** tensor. Convert integers first with `.float()` or `.type(torch.float32)`.

In [ ]:
x = torch.arange(0, 100, 10)
print(x)

print(f"Min:  {x.min()}")
print(f"Max:  {x.max()}")
print(f"Mean: {x.float().mean()}")   # x.mean() would error on int64
print(f"Sum:  {x.sum()}")

# Function-style equivalents
print(torch.min(x), torch.max(x), torch.mean(x.float()), torch.sum(x))

### Positional min/max: `argmin` / `argmax`
These return the **index** of the min or max, not the value. This is **very** common for classification: `pred_label = logits.argmax(dim=1)`.

In [ ]:
t = torch.arange(10, 100, 10)
print(t)
print("Index of max:", t.argmax().item(), "| value:", t[t.argmax()].item())
print("Index of min:", t.argmin().item(), "| value:", t[t.argmin()].item())

### Understanding `dim` (added: very important)
Without `dim`, an aggregation reduces the **whole** tensor to a single value. With `dim=k`, it reduces **along** axis `k`, and that axis **disappears** from the output shape.

For a matrix of shape `[rows, cols]`:

| Call | Collapses | Result shape | Plain English |
|---|---|---|---|
| `X.sum()` | everything | `[]` | Grand total |
| `X.sum(dim=0)` | rows | `[cols]` | One value **per column** |
| `X.sum(dim=1)` | columns | `[rows]` | One value **per row** |
| `X.sum(dim=1, keepdim=True)` | columns | `[rows, 1]` | Per row, keeps the axis (useful for broadcasting back) |

**Classification example:** logits have shape `[batch, num_classes]`, so `logits.argmax(dim=1)` gives one predicted class per sample.

In [ ]:
X = torch.tensor([[1., 2., 3.],
                  [4., 5., 6.]])   # shape [2, 3]

print("sum():          ", X.sum())
print("sum(dim=0):     ", X.sum(dim=0), "-> per column")
print("sum(dim=1):     ", X.sum(dim=1), "-> per row")
print("sum(dim=1, keepdim=True):\n", X.sum(dim=1, keepdim=True))

# Classification-style example
logits = torch.tensor([[0.1, 2.5, 0.3],    # sample 0 -> class 1
                       [1.9, 0.2, 0.4]])   # sample 1 -> class 0
print("Predicted classes:", logits.argmax(dim=1))

### Changing dtype
Use `.type(dtype)`, `.to(dtype)`, or shortcuts such as `.float()`, `.long()`, `.half()`.

In [ ]:
t = torch.arange(10., 100., 10.)
print(t.dtype)
print(t.type(torch.float16))
print(t.to(torch.int8))
print(t.long().dtype)

**TL;DR:** `min/max/mean/sum` aggregate, and `argmax/argmin` give positions. `dim=k` collapses axis `k`. `keepdim=True` keeps it as size 1. `mean` needs floats.

## 11. Reshaping, stacking, squeezing, permuting 🔑

These change a tensor's **shape** or **layout** without changing its values. You'll use them constantly to make shapes line up between layers.

| Method | What it does | Copy or view? |
|---|---|---|
| `t.reshape(shape)` | Changes shape (the element count must stay the same). Use `-1` to infer one dim. | View if possible, else a copy |
| `t.view(shape)` | Changes shape. Needs a **contiguous** tensor. | Always a view (**shares memory**) |
| `torch.stack(tensors, dim)` | Joins equal-shaped tensors along a **new** dim | Copy |
| `torch.cat(tensors, dim)` | Joins tensors along an **existing** dim | Copy |
| `t.squeeze(dim=None)` | Removes dims of size 1 | View |
| `t.unsqueeze(dim)` | Adds a size-1 dim at `dim` | View |
| `t.permute(*dims)` | Reorders dims, e.g. HWC → CHW | View (non-contiguous) |
| `t.flatten(start_dim)` | Collapses dims into one | View if possible |

> **View** = a new "window" on the **same memory**, so changing one changes the other. **Copy** = independent data.

In [ ]:
x = torch.arange(1., 8.)
print(x, x.shape)

x_reshaped = x.reshape(1, 7)
print("reshape(1, 7):", x_reshaped.shape)

print("reshape(-1, 1):", x.reshape(-1, 1).shape, "<- -1 means 'work it out'")

### `view` shares memory
Changing the view changes the original.

In [ ]:
z = x.view(1, 7)
z[:, 0] = 99
print("z:", z)
print("x:", x, "<- x changed too!")

### `stack` vs `cat` (added)
- `stack` **adds a new dimension**: four `[7]` tensors become `[4, 7]`, or `[7, 4]` with `dim=1`.
- `cat` **glues along an existing dimension**: four `[7]` tensors become `[28]`.

In [ ]:
x = torch.arange(1., 8.)

print("stack dim=0:", torch.stack([x, x, x, x], dim=0).shape)
print("stack dim=1:", torch.stack([x, x, x, x], dim=1).shape)
print("cat   dim=0:", torch.cat([x, x, x, x], dim=0).shape)

a = torch.zeros(2, 3)
b = torch.ones(2, 3)
print("cat 2-D dim=0:", torch.cat([a, b], dim=0).shape, "| dim=1:", torch.cat([a, b], dim=1).shape)

### `squeeze` / `unsqueeze`
- `squeeze()` removes size-1 dims, e.g. `[1, 7]` → `[7]`.
- `unsqueeze(dim)` adds one. The classic use is adding a **batch dimension** before passing a single sample to a model: `[3, 224, 224]` → `[1, 3, 224, 224]`.

In [ ]:
x_reshaped = torch.arange(1., 8.).reshape(1, 7)
x_squeezed = x_reshaped.squeeze()
print(f"{x_reshaped.shape} -> squeeze -> {x_squeezed.shape}")

x_unsqueezed = x_squeezed.unsqueeze(dim=0)
print(f"{x_squeezed.shape} -> unsqueeze(0) -> {x_unsqueezed.shape}")

image = torch.rand(3, 224, 224)
batch_of_one = image.unsqueeze(0)
print(f"Single image {image.shape} -> batch {batch_of_one.shape}")

### `permute`: reorder axes
`permute(2, 0, 1)` means "new dim 0 = old dim 2, new dim 1 = old dim 0, new dim 2 = old dim 1". This is the standard HWC → CHW conversion.

In [ ]:
x_hwc = torch.rand(size=(224, 224, 3))      # e.g. image loaded with PIL / OpenCV
x_chw = x_hwc.permute(2, 0, 1)              # what PyTorch models expect

print(f"HWC {x_hwc.shape} -> CHW {x_chw.shape}")

# permute returns a view: same memory
x_hwc[0, 0, 0] = 12345.
print("Shared memory?", x_chw[0, 0, 0].item() == 12345.)

### Contiguity (why `view` sometimes fails)
After `permute` or `.T`, the data in memory is no longer laid out in row order, so the tensor is **non-contiguous**. `view()` then fails. Use `reshape()` instead, or call `.contiguous()` first.

In [ ]:
print("Contiguous after permute?", x_chw.is_contiguous())

try:
    x_chw.view(-1)
except RuntimeError as e:
    print("❌ view failed:", str(e)[:80], "...")

print("reshape works:", x_chw.reshape(-1).shape)
print(".contiguous().view works:", x_chw.contiguous().view(-1).shape)

**TL;DR:** use `reshape` by default. Use `unsqueeze(0)` to add a batch dim, `permute` to reorder axes (HWC↔CHW), `stack` for a new dim and `cat` for an existing one. Views share memory.

## 12. Indexing and boolean masks

Indexing works like Python lists and NumPy, from the **outer** dimension to the **inner** one.

| Syntax | Meaning |
|---|---|
| `x[0]` | First element of dim 0 |
| `x[0, 1]` | Same as `x[0][1]` |
| `x[:, 0]` | All of dim 0, index 0 of dim 1 |
| `x[..., -1]` | Last element of the **last** dim, whatever the number of dims |
| `x[1:3]` | Slice: indices 1 and 2 |
| `x[x > 5]` | Boolean mask: elements where the condition is true (returns 1-D) |

In [ ]:
x = torch.arange(1, 10).reshape(1, 3, 3)
print(x, x.shape)

print("x[0]:\n", x[0])
print("x[0][0]:", x[0][0])
print("x[0][0][0]:", x[0][0][0])

In [ ]:
print("x[:, 0]      ->", x[:, 0])        # all of dim 0, first row
print("x[:, :, 1]   ->", x[:, :, 1])     # middle column
print("x[:, 1, 1]   ->", x[:, 1, 1])     # centre value
print("x[0, 0, :]   ->", x[0, 0, :])     # same as x[0][0]
print("x[..., -1]   ->", x[..., -1])     # last column, via ellipsis

### Boolean masks (added)
Comparisons produce `bool` tensors that you can use to **filter** or **modify** values. This is useful for thresholding predictions, masking padding, and so on.

In [ ]:
t = torch.tensor([3, -1, 7, 0, -5, 9])

mask = t > 0
print("Mask:     ", mask)
print("Positives:", t[mask])

t_relu = t.clone()
t_relu[t_relu < 0] = 0          # manual ReLU!
print("ReLU'd:   ", t_relu)

print("torch.where:", torch.where(t > 0, t, torch.zeros_like(t)))

## 13. PyTorch ↔ NumPy (and memory sharing)

| Direction | Code | Notes |
|---|---|---|
| NumPy → PyTorch | `torch.from_numpy(arr)` | **Shares memory** with `arr`; keeps NumPy's dtype (often `float64`!) |
| NumPy → PyTorch (copy) | `torch.tensor(arr)` | Independent copy |
| PyTorch → NumPy | `t.numpy()` | **Shares memory**; CPU tensors only |
| GPU / grad tensor → NumPy | `t.detach().cpu().numpy()` | The safe universal pattern |

> ⚠️ **Precision gotcha:** NumPy defaults to `float64`, but PyTorch models expect `float32`. Use `torch.from_numpy(arr).float()` (or `.type(torch.float32)`).

In [ ]:
array = np.arange(1.0, 8.0)
tensor = torch.from_numpy(array)
print(array.dtype, "->", tensor.dtype, " <- float64 carried over!")

tensor_f32 = torch.from_numpy(array).float()
print("Converted:", tensor_f32.dtype)

### Memory sharing: reassignment vs in-place
- `array = array + 1` creates a **new** array, so the tensor is unaffected.
- `array += 1` changes the array **in place**, so the tensor *also* changes because they share memory.

In [ ]:
array = np.arange(1.0, 4.0)
tensor = torch.from_numpy(array)

array = array + 1            # new object
print("After reassignment -> array:", array, "| tensor:", tensor)

array2 = np.arange(1.0, 4.0)
tensor2 = torch.from_numpy(array2)
array2 += 1                  # in-place
print("After in-place     -> array:", array2, "| tensor:", tensor2, "<- changed!")

In [ ]:
tensor = torch.ones(5)
numpy_arr = tensor.numpy()
print(tensor, numpy_arr, numpy_arr.dtype)

tensor.add_(1)               # in-place on the tensor
print("NumPy array also changed:", numpy_arr)

**TL;DR:** `from_numpy` and `.numpy()` share memory, so in-place edits affect both. Convert NumPy's `float64` to `float32`. Use `t.detach().cpu().numpy()` for GPU tensors or tensors that track gradients.

## 14. Reproducibility (random seeds)

Neural networks rely on randomness: weight initialisation, data shuffling, dropout. But computers produce **pseudo-random** numbers, which are fully determined by a starting value called a **seed**.

Setting a seed means you (or someone else) can **re-run an experiment and get the same numbers**. That matters for debugging and for comparing experiments fairly.

In [ ]:
A = torch.rand(3, 4)
B = torch.rand(3, 4)
print("Without seed, equal?", torch.equal(A, B))

In [ ]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
C = torch.rand(3, 4)

torch.manual_seed(RANDOM_SEED)    # reset before each call you want to match
D = torch.rand(3, 4)

print("With same seed, equal?", torch.equal(C, D))

> The seed sets the **starting point** of the random sequence. After one `rand()` call, the generator has moved on, so you must reset the seed to get the same numbers again.

### A reusable "seed everything" helper (added)
In real projects you'll seed every source of randomness at once:

In [ ]:
import random

def set_seed(seed: int = 42) -> None:
    # Seed Python, NumPy and PyTorch (CPU + all GPUs) for reproducible experiments.
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)   # safe to call even without a GPU

set_seed(42)
print(torch.rand(2))

> Full determinism on GPU can also need `torch.backends.cudnn.deterministic = True` and `torch.use_deterministic_algorithms(True)`, at some cost in speed. See the [PyTorch randomness docs](https://pytorch.org/docs/stable/notes/randomness.html).

## 15. Devices: CPU, CUDA, MPS 🔑

GPUs run thousands of simple operations **in parallel**, which suits the matrix multiplications that dominate deep learning. Training on a GPU is often **10–100× faster** than on a CPU.

| Device string | Hardware | Check with |
|---|---|---|
| `"cpu"` | Any CPU (always available) | – |
| `"cuda"` / `"cuda:0"` | NVIDIA GPU (CUDA) | `torch.cuda.is_available()` |
| `"mps"` | Apple Silicon GPU (M1/M2/M3/M4) | `torch.backends.mps.is_available()` |

### Ways to get a GPU

| Option | Setup effort | Pros | Cons |
|---|---|---|---|
| **Google Colab / Kaggle** | Easy | Free tier, no setup, shareable | Session limits, outputs aren't saved, limited GPU time |
| **Your own GPU** | Medium | Full control, no hourly cost, works offline | Upfront hardware cost; driver/CUDA setup |
| **Cloud (AWS, GCP, Azure, Lambda, RunPod…)** | Medium–hard | Scales to large GPUs on demand | Pay-per-hour, which adds up; more setup |

In a notebook you can run `!nvidia-smi` to see NVIDIA GPU details (memory, utilisation, driver/CUDA version).

### Device-agnostic code (best practice)
Write code that uses the best available device and falls back to CPU. That way it runs anywhere.

In [ ]:
if torch.cuda.is_available():
    device = "cuda"          # NVIDIA GPU
elif torch.backends.mps.is_available():
    device = "mps"           # Apple Silicon GPU
else:
    device = "cpu"

print(f"Using device: {device}")
print(f"CUDA GPUs available: {torch.cuda.device_count()}")

### Moving tensors (and models) with `.to(device)`
- `.to(device)` **returns a copy** on the target device, so **reassign it**: `t = t.to(device)`.
- *Models* move in place with `model.to(device)`, but reassigning is a harmless habit.
- **All tensors in an operation must be on the same device.** Otherwise you get `Expected all tensors to be on the same device`.

In [ ]:
tensor = torch.tensor([1, 2, 3])
print("Original:", tensor, tensor.device)

tensor_on_device = tensor.to(device)
print("Moved:   ", tensor_on_device, tensor_on_device.device)

### Moving back to CPU (e.g. for NumPy or Matplotlib)
NumPy only works with CPU memory, so calling `.numpy()` on a GPU tensor raises an error. Use `.cpu()` first. It returns a copy, and the original stays on the GPU.

In [ ]:
try:
    tensor_on_device.numpy()
    print("Worked (tensor is already on CPU)")
except TypeError as e:
    print("❌ TypeError:", str(e)[:90], "...")

tensor_back_on_cpu = tensor_on_device.cpu().numpy()
print("Back on CPU as NumPy:", tensor_back_on_cpu)
print("Original still on:", tensor_on_device.device)

**TL;DR:** pick the device once (`cuda` → `mps` → `cpu`), then `.to(device)` everything and reassign. Keep every tensor in an operation on the same device. Use `.cpu()` before `.numpy()`.

## 16. Bonus: a first look at autograd (added)

The original notebook only mentions `requires_grad`, but autograd is what makes training work, so here's a short preview.

- Set `requires_grad=True` and PyTorch **records every operation** on that tensor in a computation graph.
- Calling `.backward()` on a scalar result computes **gradients** (derivatives) for every tracked tensor, stored in `.grad`.
- Training = use those gradients to nudge the weights so the loss goes down (gradient descent).

**Example:** $y = x^2 + 3x$, so $\frac{dy}{dx} = 2x + 3$. At $x = 2$, the gradient is $7$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3*x

y.backward()          # compute dy/dx
print(f"y = {y.item()}, dy/dx at x=2 -> {x.grad.item()}  (expected 7.0)")

Two habits you'll use a lot later:
- `with torch.inference_mode():` (or `torch.no_grad()`) turns tracking off for **inference and evaluation**. It's faster and uses less memory.
- `t.detach()` gives a tensor that is no longer tracked, e.g. before converting it to NumPy.

In [ ]:
w = torch.randn(3, requires_grad=True)

with torch.inference_mode():
    out = w * 2
print("Tracks gradients inside inference_mode?", out.requires_grad)

print("As NumPy:", w.detach().numpy())

## 17. Cheat sheet

| Task | Code |
|---|---|
| Create from data | `torch.tensor([[1, 2], [3, 4]])` |
| Random uniform / normal | `torch.rand(3, 4)` / `torch.randn(3, 4)` |
| Zeros / ones / constant | `torch.zeros(2, 3)` / `torch.ones(2, 3)` / `torch.full((2, 3), 7.)` |
| Range | `torch.arange(0, 10, 2)` / `torch.linspace(0, 1, 5)` |
| Same shape as `t` | `torch.zeros_like(t)` |
| Inspect | `t.shape`, `t.dtype`, `t.device`, `t.ndim`, `t.numel()` |
| Python number | `t.item()` |
| Element-wise math | `t + 1`, `t * t`, `t / 2` |
| Matrix multiply | `A @ B` (or `torch.matmul`) |
| Transpose (2-D) | `A.T` |
| Aggregate | `t.sum()`, `t.mean()`, `t.max()`, `t.sum(dim=1)` |
| Index of max | `t.argmax(dim=1)` |
| Change dtype | `t.float()`, `t.long()`, `t.to(torch.float16)` |
| Reshape | `t.reshape(2, -1)` |
| Add / remove size-1 dim | `t.unsqueeze(0)` / `t.squeeze()` |
| Reorder dims | `t.permute(2, 0, 1)` |
| Join (new dim / existing dim) | `torch.stack([a, b])` / `torch.cat([a, b], dim=0)` |
| Filter | `t[t > 0]`, `torch.where(cond, a, b)` |
| NumPy → tensor | `torch.from_numpy(arr).float()` |
| Tensor → NumPy | `t.detach().cpu().numpy()` |
| Seed | `torch.manual_seed(42)` |
| Device | `device = "cuda" if torch.cuda.is_available() else "cpu"` |
| Move | `t = t.to(device)` |

## 19. Exercises

Try these without scrolling up. Solutions to the course exercises are in the [course repo](https://github.com/mrdbourke/pytorch-deep-learning), so try first and check after.

1. **Docs:** spend 10 minutes skimming the [`torch.Tensor` docs](https://pytorch.org/docs/stable/tensors.html) and [CUDA semantics](https://pytorch.org/docs/stable/notes/cuda.html).
2. Create a random tensor of shape `(7, 7)`.
3. Matrix-multiply it with a random tensor of shape `(1, 7)`. *(Hint: transpose.)*
4. Set the seed to `0` and repeat 2 and 3. Check you get the same result twice.
5. Find the GPU equivalent of `torch.manual_seed` and set it to `1234`.
6. Create two random `(2, 3)` tensors and send them to the GPU (if available), using seed `1234`.
7. Matrix-multiply the tensors from 6. *(Hint: shapes.)*
8. Find the max, min, and the index of each, for the output of 7.
9. Create a random tensor of shape `(1, 1, 1, 10)` with seed `7`, then remove all size-1 dims to get `(10,)`. Print both tensors and their shapes.
10. **Extra:** given a batch of HWC images `torch.rand(8, 64, 64, 3)`, convert it to NCHW and normalise each channel to mean 0 and std 1 using broadcasting.

In [ ]:
# Exercise scratch space

---
### Resources
- [PyTorch Tensors tutorial](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
- [`torch.Tensor` reference](https://pytorch.org/docs/stable/tensors.html)
- [Broadcasting semantics](https://pytorch.org/docs/stable/notes/broadcasting.html)
- [Reproducibility notes](https://pytorch.org/docs/stable/notes/randomness.html)
- [Course book chapter: learnpytorch.io](https://www.learnpytorch.io/00_pytorch_fundamentals/)

*Next up → 01 · PyTorch Workflow*